# logit-thermo 探索：チェスの盤の上のエネルギー（GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは 40〜70 分。終わったら自動で切れます。
- 探索（予想なし）。Lichess のパズル（CC0）から 5 つのテーマ × 50 局面を選び、局面を棋譜の形（FEN の見出し＋相手の 1 手）で書き、Stockfish（Stockfish）で全部の合法手のエネルギー（最善手からの勝率の下がり方）を出し、4 つのモデル（Pythia-1B・OLMo-2-1B・Qwen2.5-1.5B・Qwen2.5-7B）が各手に付ける確率と比べます。**結果は表示しません。**
- 保存先 `A2_explore/`：`chess_positions.json`（局面とエネルギー）、`chess_scores_*.json`（モデルごと）、`chess_summary.json`。途中で切れても、もう一度「すべて実行」すれば、終わったところは飛ばします。
- 計算のコード：`scripts/a2_explore_chess.py`（`4d5c4b4a3cdba37e…`）を埋め込み。Stockfishは `apt`、チェスのライブラリ（python-chess）は `pip` で入れます。Lichess のサイトにはつなぎません。
- **通知**：🚀・区切りごとの ✅・🎉・⚠️。名札は「logit-thermo 探索 チェスの盤のエネルギー（Stockfish・4 モデル、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 チェスの盤のエネルギー（Stockfish・4 モデル、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore chess energy', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import hashlib, json, math, gc
SRCCH = '"""探索：チェスの盤の上でエネルギーを測る下見（2026-10-30 10:0x）。確かめには使わない（予想なし）。GPU（L4）。\nscripts/a2_2_collect.py を実行したのと同じ名前空間で実行する（load_model だけ使う。7B は bfloat16 で自前に読む）。\n\n発想（オーナー 2026-10-29 23:32〜10-30 00:00、探索ノート 8.14）：（オーナーの承認・発言。原文は非公開の記録にある）\nStockfish（Stockfish）の最善手を基準にして、そこからの逸脱をエネルギーとして見る。\n\n材料：Lichess のパズル（Hugging Face の Lichess/chess-puzzles、CC0）。テーマ zugzwang・skewer・fork・mateIn1・quietMove を 50 局面ずつ。\n  選び方：データの並び順に読み、1000 ≤ Rating ≤ 2000、Popularity ≥ 50、NbPlays ≥ 200 のものを、CH_THEMES の順で最初に当てはまる\n  テーマに入れる（テーマの枠が埋まっていたら次のテーマ）。候補はテーマごとに CH_CAND 個まで集め、棋譜が取れたものから 50 個使う。\n局面：パズルの FEN（相手の 1 手の前）から、Moves の最初の手（相手の手）を指した局面。その局面で手番の側が指す。\n見せ方：棋譜の形（PGN）で、見出しに [SetUp "1"] と [FEN "<相手の手の前の局面>"] を書き、相手の 1 手（SAN、手番号つき）を続ける。\n  白番なら最後に「 N.」を足す。（はじめは Lichess から元の対局の棋譜を取り寄せる設計だったが、2026-10-30 の Colab からは 3 回とも\n  HTML のページしか返らなかったので、ネットにつながずに作れるこの形に替えた。）\nエネルギー：Stockfish（深さ CH_DEPTH、全部の合法手を MultiPV）で各手の評価値 → Lichess の式で勝率（0〜1、手番側から見て）→\n  E ＝ 勝率（最善手）− 勝率（その手）。パス：王手でなければ、手番だけ相手に渡した局面の評価値 → E_pass（負なら「パスできれば最善手より良い」）。\nモデルの確率：各合法手の SAN の前に空白を付けた文字列の log 確率（トークンの和）。合法手だけで足して 1 にしたものを p、足す前の和を「合法手の質量」。\n  （O-O と O-O-O は文字列として前後の関係にあるので、両方合法な局面は記録する。）\n"""\nimport io, json, math, os, re, time\nimport numpy as np\n\nCH_THEMES = [\'zugzwang\', \'skewer\', \'fork\', \'mateIn1\', \'quietMove\']\nCH_N, CH_CAND = 50, 70\nCH_RATING, CH_POP, CH_PLAYS = (1000, 2000), 50, 200\nCH_DEPTH = 12\nCH_MODELS = [(\'pythia-1b\', \'EleutherAI/pythia-1b\', \'float32\'), (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\', \'float32\'),\n             (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\', \'float32\'), (\'qwen2.5-7b\', \'Qwen/Qwen2.5-7B\', \'bfloat16\')]\nCH_HEADER = \'[Event "Rated game"]\\n[Site "https://lichess.org"]\\n[Result "*"]\\n\\n\'\nCH_TGRID = np.exp(np.linspace(math.log(1e-3), math.log(10.0), 400))\nCH_CHUNK = 4\n\n\n# ---------- 勝率とエネルギー（numpy だけ） ----------\ndef win_from_cp(cp):\n    """Lichess の式：勝率（0〜1）。cp は手番側から見たセンチポーン。"""\n    return 0.5 + 0.5 * (2.0 / (1.0 + math.exp(-0.00368208 * cp)) - 1.0)\n\n\ndef win_from_score(kind, val):\n    """kind：\'cp\' か \'mate\'。mate は手番側から見て正なら勝ち（1）、負なら負け（0）。"""\n    if kind == \'mate\':\n        return 1.0 if val > 0 else 0.0\n    return win_from_cp(val)\n\n\n# ---------- 当てはめ（numpy だけ） ----------\ndef boltzmann(E, T):\n    z = -np.asarray(E, float) / T\n    z = z - z.max()\n    q = np.exp(z)\n    return q / q.sum()\n\n\ndef fit_T(Ps, Es):\n    """局面ごとの (p, E) の組のリスト → 平均 KL(p ‖ q_T) が最小になる T と、そのときの KL、一様分布との KL。"""\n    best = (float(\'inf\'), float(\'nan\'))\n    kl_unif = float(np.mean([float(np.sum(p * (np.log(np.clip(p, 1e-300, None)) - math.log(1.0 / len(p))))) for p in Ps]))\n    for T in CH_TGRID:\n        kl = 0.0\n        for p, E in zip(Ps, Es):\n            q = boltzmann(E, T)\n            kl += float(np.sum(p * (np.log(np.clip(p, 1e-300, None)) - np.log(np.clip(q, 1e-300, None)))))\n        kl /= len(Ps)\n        if kl < best[0]:\n            best = (kl, float(T))\n    return dict(T=best[1], KL=best[0], KL_uniform=kl_unif)\n\n\ndef spearman_np(a, b):\n    a, b = np.asarray(a, float), np.asarray(b, float)\n    if len(a) < 3 or np.all(a == a[0]) or np.all(b == b[0]):\n        return float(\'nan\')\n    ra = np.argsort(np.argsort(a)).astype(float); rb = np.argsort(np.argsort(b)).astype(float)\n    # 同じ値の順位は平均（小さい組なので素直に）\n    for r, x in ((ra, a), (rb, b)):\n        for v in np.unique(x):\n            m = x == v\n            r[m] = r[m].mean()\n    return float(np.corrcoef(ra, rb)[0, 1])\n\n\ndef summarize(positions, scores):\n    """positions：局面のリスト（moves_san・E・theme・E_pass）、scores：{モデル: [局面ごとの log 確率の並び（moves_san の順）]}。"""\n    out = {\'n_positions\': len(positions), \'themes\': CH_THEMES, \'models\': {}}\n    for m, S in scores.items():\n        per_theme = {}\n        for th in CH_THEMES + [\'all\']:\n            idx = [i for i, pz in enumerate(positions) if th == \'all\' or pz[\'theme\'] == th]\n            if not idx:\n                continue\n            Ps, Es, mass, rho, top1, ent, pbest = [], [], [], [], [], [], []\n            for i in idx:\n                lp = np.asarray(S[i], float)\n                E = np.asarray(positions[i][\'E\'], float)\n                mass.append(float(np.exp(lp).sum()))\n                p = np.exp(lp - lp.max()); p /= p.sum()\n                Ps.append(p); Es.append(E)\n                rho.append(spearman_np(-lp, E))                     # 正なら「確率の低い手ほどエネルギーが高い」\n                top1.append(float(E[int(np.argmax(p))] <= 0.01))\n                ent.append(float(-(p * np.log(np.clip(p, 1e-300, None))).sum()))\n                pbest.append(float(p[E <= 0.01].sum()))\n            ft = fit_T(Ps, Es)\n            per_theme[th] = dict(n=len(idx), legal_mass_mean=float(np.mean(mass)), legal_mass_median=float(np.median(mass)),\n                                 rho_median=float(np.nanmedian(rho)), rho_pos_frac=float(np.nanmean(np.array(rho) > 0)),\n                                 top1_best=float(np.mean(top1)), p_on_best=float(np.mean(pbest)), entropy=float(np.mean(ent)),\n                                 entropy_uniform=float(np.mean([math.log(len(p)) for p in Ps])), **ft)\n        out[\'models\'][m] = per_theme\n    # 局面の側の記録（モデルによらない）\n    for th in CH_THEMES:\n        idx = [i for i, pz in enumerate(positions) if pz[\'theme\'] == th]\n        Ep = [positions[i][\'E_pass\'] for i in idx if positions[i][\'E_pass\'] is not None]\n        gap = [sorted(positions[i][\'E\'])[1] if len(positions[i][\'E\']) > 1 else float(\'nan\') for i in idx]\n        nb = [int(np.sum(np.asarray(positions[i][\'E\']) <= 0.01)) for i in idx]\n        out.setdefault(\'positions\', {})[th] = dict(n=len(idx), n_legal_mean=float(np.mean([len(positions[i][\'E\']) for i in idx])),\n                                                   gap_median=float(np.nanmedian(gap)), n_near_best_mean=float(np.mean(nb)),\n                                                   E_pass_median=(float(np.median(Ep)) if Ep else None),\n                                                   pass_better_frac=(float(np.mean(np.array(Ep) < 0)) if Ep else None),\n                                                   n_pass=len(Ep))\n    return out\n\n\n# ---------- チェス（python-chess と Stockfish。Colab で使う） ----------\ndef puzzle_rows(limit_scan=3_000_000):\n    from datasets import load_dataset\n    ds = load_dataset(\'Lichess/chess-puzzles\', split=\'train\', streaming=True)\n    got = {t: [] for t in CH_THEMES}\n    for k, r in enumerate(ds):\n        if k >= limit_scan or all(len(v) >= CH_CAND for v in got.values()):\n            break\n        th = r[\'Themes\'].split() if isinstance(r[\'Themes\'], str) else list(r[\'Themes\'])\n        if not (CH_RATING[0] <= r[\'Rating\'] <= CH_RATING[1] and r[\'Popularity\'] >= CH_POP and r[\'NbPlays\'] >= CH_PLAYS and r.get(\'GameId\')):\n            continue\n        for t in CH_THEMES:\n            if t in th:\n                if len(got[t]) < CH_CAND:\n                    got[t].append(dict(PuzzleId=r[\'PuzzleId\'], GameId=r[\'GameId\'], FEN=r[\'FEN\'], Moves=r[\'Moves\'],\n                                       Rating=int(r[\'Rating\']), Themes=th, theme=t))\n                break                                                # テーマは順番で最初に当てはまる 1 つだけ\n    return got\n\n\ndef build_position_fen(pz):\n    """パズルの FEN（相手の手の前）と相手の 1 手から、局面とプロンプトを作る。ネットは使わない。"""\n    import chess\n    b0 = chess.Board(pz[\'FEN\'])\n    first = chess.Move.from_uci(pz[\'Moves\'].split()[0])\n    if first not in b0.legal_moves:\n        return None\n    san_first = b0.san(first)\n    n = b0.fullmove_number\n    prefix = f\'{n}. {san_first}\' if b0.turn == chess.WHITE else f\'{n}... {san_first}\'\n    b = b0.copy(); b.push(first)\n    head = CH_HEADER.replace(\'[Result "*"]\\n\', \'[Result "*"]\\n[SetUp "1"]\\n[FEN "\' + pz[\'FEN\'] + \'"]\\n\')\n    prompt = head + prefix + (f\' {b.fullmove_number}.\' if b.turn == chess.WHITE else \'\')\n    legal = list(b.legal_moves)\n    sans = [b.san(m) for m in legal]\n    both_castle = (\'O-O\' in sans) and (\'O-O-O\' in sans)\n    return dict(PuzzleId=pz[\'PuzzleId\'], theme=pz[\'theme\'], Rating=pz[\'Rating\'], prompt=prompt, fen=b.fen(),\n                moves_uci=[m.uci() for m in legal], moves_san=sans, solution_uci=pz[\'Moves\'].split()[1],\n                both_castle=both_castle)\n\n\ndef engine_energies(engine, fen, depth=CH_DEPTH):\n    """→ ({uci: E}、E_pass か None、最善手の勝率)。E_pass は、王手でなく、手番を渡した局面が正しい局面のときだけ出す。"""\n    import chess, chess.engine\n    b = chess.Board(fen)\n    n = b.legal_moves.count()\n    infos = engine.analyse(b, chess.engine.Limit(depth=depth), multipv=n)\n    W = {}\n    for inf in infos:\n        sc = inf[\'score\'].pov(b.turn)\n        W[inf[\'pv\'][0].uci()] = win_from_score(\'mate\', sc.mate()) if sc.is_mate() else win_from_score(\'cp\', sc.score())\n    wbest = max(W.values())\n    Epass = None\n    if not b.is_check():\n        b2 = b.copy(stack=False)\n        b2.turn = not b.turn; b2.ep_square = None                    # 手番だけ相手に渡す（null move は送らない）\n        if b2.is_valid():\n            inf = engine.analyse(b2, chess.engine.Limit(depth=depth))\n            sc = inf[\'score\'].pov(b.turn)\n            wpass = win_from_score(\'mate\', sc.mate()) if sc.is_mate() else win_from_score(\'cp\', sc.score())\n            Epass = wbest - wpass\n    return {u: wbest - w for u, w in W.items()}, Epass, wbest\n\n\ndef collect_positions(engine, notify=print):\n    rows = puzzle_rows()\n    positions, skipped = [], {\'cut\': 0, \'engine\': 0}\n    for t in CH_THEMES:\n        k = 0\n        for pz in rows[t]:\n            if k >= CH_N:\n                break\n            pos = build_position_fen(pz)\n            if pos is None:\n                skipped[\'cut\'] += 1; continue\n            Ed, Ep, wb = engine_energies(engine, pos[\'fen\'])\n            if set(Ed) != set(pos[\'moves_uci\']):\n                skipped[\'engine\'] += 1; continue\n            pos[\'E\'] = [float(Ed[u]) for u in pos[\'moves_uci\']]\n            pos[\'E_pass\'] = None if Ep is None else float(Ep)\n            pos[\'W_best\'] = float(wb)\n            positions.append(pos); k += 1\n        notify(f\'局面 {t}：{k} 個（候補 {len(rows[t])}）\')\n    assert len(positions) >= 100, (\'局面がほとんど集まりませんでした\', skipped)\n    return positions, skipped, {t: len(v) for t, v in rows.items()}\n\n\n# ---------- モデルの確率（Colab で使う） ----------\ndef load_any(hf_id, dtype, device):\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=getattr(torch, dtype)).to(device).eval()\n    return tok, model\n\n\ndef score_moves(tok, model, prompt, sans, device):\n    """→ 各手の log 確率（トークンの和、float64）。並べ方は sans の順。プロンプトと手は別々にトークン化してつなぐ。"""\n    import torch\n    bos = [tok.bos_token_id] if (tok.bos_token_id is not None and getattr(tok, \'add_bos_token\', False)) else []\n    P = bos + tok(prompt, add_special_tokens=False).input_ids\n    C = [tok(\' \' + s, add_special_tokens=False).input_ids for s in sans]\n    out = np.zeros(len(sans))\n    pad = tok.pad_token_id if tok.pad_token_id is not None else (tok.eos_token_id or 0)\n    with torch.no_grad():\n        for a in range(0, len(C), CH_CHUNK):\n            cc = C[a:a + CH_CHUNK]\n            L = len(P) + max(len(c) for c in cc)\n            ids = torch.full((len(cc), L), pad, dtype=torch.long)\n            att = torch.zeros((len(cc), L), dtype=torch.long)\n            for j, c in enumerate(cc):\n                seq = P + c\n                ids[j, :len(seq)] = torch.tensor(seq); att[j, :len(seq)] = 1\n            lg = model(input_ids=ids.to(device), attention_mask=att.to(device)).logits.float()\n            lp = torch.log_softmax(lg, -1)\n            for j, c in enumerate(cc):\n                pos = torch.arange(len(P) - 1, len(P) - 1 + len(c), device=device)\n                out[a + j] = float(lp[j, pos, torch.tensor(c, device=device)].double().sum())\n    return out\n\n\ndef self_test_chess_math():\n    assert abs(win_from_cp(0) - 0.5) < 1e-12 and win_from_cp(300) > 0.7 and win_from_cp(-300) < 0.3\n    assert win_from_score(\'mate\', 3) == 1.0 and win_from_score(\'mate\', -2) == 0.0\n    # 当てはめ：q_T そのものを p にすると、その T が戻る\n    rng = np.random.default_rng(0)\n    Es = [np.sort(rng.uniform(0, 1, 20)) for _ in range(30)]\n    for E in Es:\n        E -= E.min()\n    Ps = [boltzmann(E, 0.05) for E in Es]\n    ft = fit_T(Ps, Es)\n    assert abs(math.log(ft[\'T\'] / 0.05)) < 0.05 and ft[\'KL\'] < 1e-3, ft\n    assert abs(spearman_np([1, 2, 3, 4], [10, 20, 30, 40]) - 1) < 1e-12 and abs(spearman_np([1, 2, 3, 4], [4, 3, 2, 1]) + 1) < 1e-12\n    # summarize：一様な確率なら top1 は偶然、rho は nan か小さい\n    positions = [dict(theme=CH_THEMES[i % 5], E=list(E), E_pass=(0.1 if i % 2 else None)) for i, E in enumerate(Es)]\n    scores = {\'boltz\': [np.log(boltzmann(E, 0.05)) - 0.3 for E in Es]}\n    s = summarize(positions, scores)\n    a = s[\'models\'][\'boltz\'][\'all\']\n    assert abs(a[\'legal_mass_mean\'] - math.exp(-0.3)) < 1e-9 and a[\'rho_median\'] > 0.99 and abs(math.log(a[\'T\'] / 0.05)) < 0.05, a\n    return \'OK\'\n\n\ndef self_test_chess_board():\n    """python-chess と Stockfish が入っていること、局面とプロンプトの作り方、メイトの局面のエネルギー。"""\n    import chess, chess.engine\n    # 3... Nf6 の前（黒番、3 手目）の局面を「パズルの FEN」とし、相手の手 Nf6（g8f6）のあとで白が指す（答えは Qxf7#）\n    b = chess.Board(); [b.push_san(s) for s in [\'e4\', \'e5\', \'Bc4\', \'Nc6\', \'Qh5\']]\n    pz = dict(PuzzleId=\'t\', theme=\'mateIn1\', Rating=1000, FEN=b.fen(), Moves=\'g8f6 h5f7\')\n    pos = build_position_fen(pz)\n    assert pos is not None and pos[\'prompt\'].endswith(\'3... Nf6 4.\'), pos and pos[\'prompt\']\n    assert (\'[FEN "\' + b.fen() + \'"]\') in pos[\'prompt\'] and \'[SetUp "1"]\' in pos[\'prompt\']\n    assert \'Qxf7#\' in pos[\'moves_san\'] and pos[\'solution_uci\'] == \'h5f7\'\n    # 白番の前（白が相手）：1. e4 のあと黒が指す → プロンプトは「1. e4」で終わる\n    pz2 = dict(PuzzleId=\'t2\', theme=\'fork\', Rating=1000, FEN=chess.STARTING_FEN, Moves=\'e2e4 e7e5\')\n    pos2 = build_position_fen(pz2)\n    assert pos2[\'prompt\'].endswith(\'\\n\\n1. e4\') and \'e5\' in pos2[\'moves_san\'], pos2[\'prompt\']\n    assert build_position_fen(dict(pz, Moves=\'a1a8 h5f7\')) is None                 # 指せない手なら飛ばす\n    eng = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)\n    try:\n        Ed, Ep, wb = engine_energies(eng, pos[\'fen\'], depth=8)\n        assert Ed[\'h5f7\'] == 0.0 and wb == 1.0 and all(v > 0 for u, v in Ed.items() if u != \'h5f7\'), Ed\n        assert set(Ed) == set(pos[\'moves_uci\'])\n    finally:\n        eng.quit()\n    return \'OK\'\n\n\ndef self_test_score(device):\n    """小さなランダムのモデルで：まとめて（右を埋めて）計算した手の log 確率が、1 本ずつ計算したものと同じ。"""\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    prompt = CH_HEADER + \'1. e4 e5 2. Nf3 Nc6 3.\'\n    sans = [\'Bb5\', \'Bc4\', \'d4\', \'O-O\', \'Nxe5\', \'Qe2\', \'h3\', \'a4\', \'c3\']\n    a = score_moves(tok, model, prompt, sans, device)\n    one = np.array([score_moves(tok, model, prompt, [s_], device)[0] for s_ in sans])\n    assert np.allclose(a, one, atol=1e-4), (a, one)\n    # 手で 1 本だけ計算\n    P = tok(prompt, add_special_tokens=False).input_ids; c = tok(\' Bb5\', add_special_tokens=False).input_ids\n    with torch.no_grad():\n        lp = torch.log_softmax(model(torch.tensor([P + c], device=device)).logits.float(), -1)[0]\n    ref = sum(float(lp[len(P) - 1 + k, c[k]]) for k in range(len(c)))\n    assert abs(ref - a[0]) < 1e-4, (ref, a[0])\n    return \'OK\'\n'
assert hashlib.sha256(SRCCH.encode()).hexdigest() == '4d5c4b4a3cdba37ef27894bdc9281c2ffbd0a0679e27db5e721c53a48aac71f6'
exec(SRCCH)
def _clean(o):
    if isinstance(o, dict): return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, float) and not math.isfinite(o): return None
    return o
print('コードを置きました')


In [ ]:
# セル3：道具を入れる → 自己テスト → 局面とエネルギー → モデルの確率 → まとめ（結果は表示しない）
import subprocess, shutil, sys
try:
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'stockfish'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'chess'], check=True)
    STOCKFISH_PATH = shutil.which('stockfish') or '/usr/games/stockfish'
    print('自己テスト：', self_test_chess_math(), self_test_chess_board(), self_test_score('cuda'))
    OUTD = '/content/drive/MyDrive/logit-thermo' + '/A2_explore'
    os.makedirs(OUTD, exist_ok=True)
    pp = OUTD + '/chess_positions.json'
    if os.path.exists(pp):
        P = json.load(open(pp))
    else:
        import chess.engine
        eng = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
        eng.configure({'Threads': 2, 'Hash': 256})
        positions, skipped, ncand = collect_positions(eng, notify=lambda s: notify('✅ ' + s, tags='white_check_mark'))
        P = dict(positions=positions, skipped=skipped, n_cand=ncand, engine=eng.id.get('name'), depth=CH_DEPTH)
        eng.quit()
        json.dump(P, open(pp + '.part', 'w'), ensure_ascii=False); os.replace(pp + '.part', pp)
    for name, hf_id, dt in CH_MODELS:
        sp = OUTD + f'/chess_scores_{name}.json'
        if os.path.exists(sp):
            continue
        t0 = time.time()
        tok, model = load_any(hf_id, dt, 'cuda')
        S = [score_moves(tok, model, pz['prompt'], pz['moves_san'], 'cuda').tolist() for pz in P['positions']]
        json.dump(dict(model=hf_id, dtype=dt, scores=S), open(sp + '.part', 'w')); os.replace(sp + '.part', sp)
        del model; gc.collect(); torch.cuda.empty_cache()
        notify(f'✅ {name} おわり（{time.time() - t0:.0f} 秒）', tags='white_check_mark')
    scores = {name: json.load(open(OUTD + f'/chess_scores_{name}.json'))['scores'] for name, _, _ in CH_MODELS}
    summ = summarize(P['positions'], scores)
    summ.update(engine=P['engine'], depth=P['depth'], skipped=P['skipped'], n_cand=P['n_cand'],
                both_castle=int(sum(pz['both_castle'] for pz in P['positions'])),
                dtypes={name: dt for name, _, dt in CH_MODELS}, a2_explore_chess_sha256='4d5c4b4a3cdba37ef27894bdc9281c2ffbd0a0679e27db5e721c53a48aac71f6')
    json.dump(_clean(summ), open(OUTD + '/chess_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 チェスの盤のエネルギーで止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：チェスの盤のエネルギーを A2_explore/chess_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
